In [33]:
import tensorflow as tf
from tensorflow import keras 
from keras.models import Sequential
from keras.layers import Flatten, Dense
from keras.applications.mobilenet_v2 import MobileNetV2

In [34]:
conv_base = MobileNetV2(
    include_top = False,
    weights = 'imagenet',
    input_shape = (96,96,3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [35]:
leaf_model = Sequential()
leaf_model.add(conv_base)
leaf_model.add(Flatten())
leaf_model.add(Dense(1, activation='sigmoid'))

In [36]:
leaf_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 11520)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │        11,521 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,269,505 (8.66 MB)

 Trainable params: 2,235,393 (8.53 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [37]:
conv_base.trainable = False

In [38]:
dataset = keras.preprocessing.image_dataset_from_directory(
    directory='./Dataset',
    shuffle=True,
    batch_size=16,
    image_size=(96, 96),  # Resize all images to 96x96
    interpolation='bilinear'  # Use bilinear interpolation for smooth resizing
)

Found 71920 files belonging to 2 classes.


In [39]:
leaf_class_names = dataset.class_names

In [40]:
leaf_class_names

['Leaf', 'Not Leaf']

In [41]:
dataset.cardinality().numpy()

4495

In [42]:
train_ds = dataset.take(3146)
valid_ds = dataset.skip(3146).take(899)
test_ds = dataset.skip(4045)

In [43]:
print(train_ds.cardinality())
print(valid_ds.cardinality())
print(test_ds.cardinality())

tf.Tensor(3146, shape=(), dtype=int64)
tf.Tensor(899, shape=(), dtype=int64)
tf.Tensor(450, shape=(), dtype=int64)


In [44]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)
valid_ds = valid_ds.cache().shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)

In [45]:
leaf_model.compile(
    optimizer = 'adam',
    metrics = ['accuracy'],
    loss = 'binary_crossentropy'
)

In [46]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', save_best_only=True)
]

In [47]:
history = leaf_model.fit(
    train_ds,
    validation_data = valid_ds,
    epochs = 50,
    callbacks=callbacks
)

Epoch 1/50


Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS

   1/3146 ━━━━━━━━━━━━━━━━━━━━ 15:25:36 18s/step - accuracy: 0.1250 - loss: 1.9678

2025-03-11 15:22:49.151154: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


   3/3146 ━━━━━━━━━━━━━━━━━━━━ 7:22 141ms/step - accuracy: 0.4444 - loss: 1.3663

...

2108/3146 ━━━━━━━━━━━━━━━━━━━━ 53s 52ms/step - accuracy: 0.9689 - loss: 0.0993

Invalid SOS parameters for sequential JPEG


3145/3146 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.9726 - loss: 0.0879

Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS

3146/3146 ━━━━━━━━━━━━━━━━━━━━ 264s 78ms/step - accuracy: 0.9726 - loss: 0.0879 - val_accuracy: 0.9872 - val_loss: 0.0482
Epoch 2/50
3146/3146 ━━━━━━━━━━━━━━━━━━━━ 136s 43ms/step - accuracy: 0.9895 - loss: 0.0381 - val_accuracy: 0.9793 - val_loss: 0.0892
Epoch 3/50
3146/3146 ━━━━━━━━━━━━━━━━━━━━ 155s 49ms/step - accuracy: 0.9891 - loss: 0.0433 - val_accuracy: 0.9784 - val_loss: 0.0892
Epoch 4/50
3146/3146 ━━━━━━━━━━━━━━━━━━━━ 151s 48ms/step - accuracy: 0.9917 - loss: 0.0314 - val_accuracy: 0.9871 - val_loss: 0.0558
Epoch 5/50
3146/3146 ━━━━━━━━━━━━━━━━━━━━ 136s 43ms/step - accuracy: 0.9927 - loss: 0.0298 - val_accuracy: 0.9854 - val_loss: 0.0878
Epoch 6/50
3146/3146 ━━━━━━━━━━━━━━━━━━━━ 135s 43ms/step - accuracy: 0.9931 - loss: 0.0285 - val_accuracy: 0.9900 - val_loss: 0.0512


In [49]:
leaf_model.save('../saved_models/leaf_model.keras')